# Kata kunci tanpa label

Keyword bebas berbeda dari token tagging. Evaluasi perlu gold keyphrase; tanpa gold laporkan audit relevansi/duplikasi.

Kode lengkap dari pipeline dan snippet asli. Ikuti urutan tahap dan ubah konfigurasi awal sesuai data. Alternatif tersedia pada cheatsheet utama.

## Setup & konfigurasi

Satu-satunya sel yang wajib diubah. Jalankan paling awal.

In [ ]:
import os
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
from pathlib import Path
import importlib.metadata as md

def env_report(models_dir=None):
    import torch
    for p in ("torch", "transformers", "scikit-learn", "pandas", "numpy", "sentence-transformers", "lightgbm"):
        try:
            print(p, md.version(p))
        except md.PackageNotFoundError:
            print(p, "TIDAK ADA")
    gpu = torch.cuda.is_available()
    print("cuda", gpu, torch.cuda.get_device_name(0) if gpu else "")
    if gpu:
        print("VRAM GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "| bf16", torch.cuda.is_bf16_supported())
    if models_dir and Path(models_dir).exists():
        print("model lokal:", sorted(p.name for p in Path(models_dir).iterdir() if p.is_dir()))
    return gpu

import os
import random
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

# ===== jalankan =====
import re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
MODELS = Path(os.environ.get("COMP_ROOT", r"C:\gemastik")) / "models"
SEED = 42
seed_everything(SEED)
env_report(MODELS)

DATA, TEXT, K = "train.csv", "text", 5


## Muat data

Baca file, otomatis mencoba encoding, simpan parquet.

In [ ]:
from pathlib import Path
import pandas as pd

def read_any(path, cache=True, **kw):
    """df = read_any('train.csv'). Lain kali otomatis membaca train.parquet."""
    p = Path(path)
    if p.suffix == ".parquet":
        return pd.read_parquet(p, **kw)
    if cache and p.with_suffix(".parquet").exists():
        return pd.read_parquet(p.with_suffix(".parquet"))
    if p.suffix in (".jsonl", ".json"):
        return pd.read_json(p, lines=p.suffix == ".jsonl", **kw)
    if p.suffix in (".xlsx", ".xls"):
        return pd.read_excel(p, **kw)
    sep = kw.pop("sep", "\t" if p.suffix == ".tsv" else ",")
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(p, sep=sep, encoding=enc, on_bad_lines="warn", **kw)
            break
        except UnicodeDecodeError:
            continue
    if cache:
        df.to_parquet(p.with_suffix(".parquet"), index=False)
    return df

# ===== jalankan =====
df = read_any(DATA)
print(df.shape)


## Preprocessing

Stopword dibuang untuk kata kunci.

In [ ]:
import html
import re
import unicodedata

SLANG = {"yg": "yang", "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "tdk": "tidak", "bgt": "banget", "tp": "tapi", "klo": "kalau", "kalo": "kalau",
         "udh": "sudah", "dah": "sudah", "dpt": "dapat", "jg": "juga", "jd": "jadi", "bnyk": "banyak", "karna": "karena", "krn": "karena", "utk": "untuk", "jgn": "jangan",
         "org": "orang", "dr": "dari", "lgi": "lagi", "lg": "lagi", "bnr": "benar", "sdh": "sudah", "dgn": "dengan", "aja": "saja"}
STOPWORDS = set("""ada adalah agar akan aku anda antara apa apakah atau bagi bahwa banyak beberapa begitu bila bisa dalam dan dari dengan di dia dua hal hanya harus ia ini itu
jadi jika juga kami kamu karena ke kepada ketika kita lagi lain lebih maka masih mereka mungkin nya oleh pada para saat saja sama sampai sangat satu saya sebagai sebuah sedang
sehingga sekali selalu semua seperti setelah sudah supaya tapi telah tentang tersebut tetapi untuk yaitu yang""".split())   # sengaja tanpa tidak, bukan, jangan, belum, kurang
NEGATION = "tidak|tak|bukan|jangan|belum|kurang|tanpa"
_MOJIBAKE = re.compile("[\u00c3\u00c2\u00e2\u00f0]")

PRESETS = {
    # keep = biarkan, token = ganti penanda, remove = hapus
    "none":        dict(unicode=False, html=False, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "tfidf":       dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="keep", emoji="keep", number="keep", lower=True, repeat=True, punct="keep", slang=False, negation=False, stopword=False),
    "social":      dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="word", emoji="text", number="keep", lower=True, repeat=True, punct="keep", slang=True, negation=False, stopword=False),
    "topic":       dict(unicode=True, html=True, url="remove", email="remove", phone="remove", mention="remove", hashtag="word", emoji="remove", number="remove", lower=True, repeat=True, punct="remove", slang=True, negation=False, stopword=True),
    "transformer": dict(unicode=True, html=True, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "indobertweet": dict(unicode=True, html=True, url="HTTPURL", email="keep", phone="keep", mention="@USER", hashtag="keep", emoji="text", number="keep", lower=True, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
}

def _sub(text, pattern, how, token):
    return text if how == "keep" else re.sub(pattern, " " if how == "remove" else f" {token if how == 'token' else how} ", text)

def preprocess(text, preset="tfidf", slang_dict=SLANG, stopwords=STOPWORDS, **override):
    """preprocess('Halo http://a.id', 'social'). Ubah satu langkah: preprocess(t, 'tfidf', emoji='remove')."""
    c = {**PRESETS[preset], **override}
    t = "" if text is None or text != text else str(text)
    if c["unicode"]:
        if _MOJIBAKE.search(t):                         # teks rusak encoding
            import ftfy
            t = ftfy.fix_text(t)
        t = unicodedata.normalize("NFKC", t).replace("​", "")
    if c["html"]:
        t = re.sub(r"<[^>]+>", " ", html.unescape(t))
    t = _sub(t, r"[\w.+-]+@[\w-]+\.[\w.-]+", c["email"], "EMAIL")          # email sebelum mention
    t = _sub(t, r"https?://\S+|www\.\S+", c["url"], "URL")
    t = _sub(t, r"(?<!\d)(?:\+?62|0)8\d{7,11}(?!\d)", c["phone"], "TELEPON")
    t = _sub(t, r"@\w+", c["mention"], "USER")
    if c["hashtag"] == "word":
        t = re.sub(r"#(\w+)", r" \1 ", t)
    elif c["hashtag"] != "keep":
        t = _sub(t, r"#\w+", c["hashtag"], "HASHTAG")
    if c["emoji"] != "keep":
        import emoji
        if c["emoji"] == "id":                          # emoji jadi kata Indonesia
            from indoNLP.preprocessing import emoji_to_words
            t = emoji_to_words(t, delimiter=(" ", " "))
        else:
            t = emoji.demojize(t, delimiters=(" ", " ")) if c["emoji"] == "text" else emoji.replace_emoji(t, " " if c["emoji"] == "remove" else " EMOJI ")
    t = _sub(t, r"\d+(?:[.,]\d+)*", c["number"], "ANGKA")
    if c["lower"]:
        keep = {m: m for m in ("HTTPURL", "@USER", "URL", "EMAIL", "TELEPON", "USER", "ANGKA", "EMOJI", "HASHTAG") if m in t}
        t = t.lower()
        for m in keep:
            t = re.sub(r"(?<!\w)" + re.escape(m.lower()) + r"(?!\w)", m, t)
    if c["repeat"]:
        t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    if c["punct"] == "remove":
        t = re.sub(r"[^\w\s]", " ", t)
    if c["slang"]:
        t = re.sub(r"\b\w+\b", lambda m: slang_dict.get(m.group(0).lower(), m.group(0)), t)
    if c["negation"]:
        t = re.sub(rf"\b({NEGATION})\s+(\w+)", r"\1_\2", t, flags=re.I)
    if c["stopword"]:
        t = " ".join(w for w in t.split() if w.lower() not in stopwords)
    return re.sub(r"\s+", " ", t).strip()

def preprocess_corpus(texts, preset="tfidf", stem=False, slang_csv=None, **override):
    """Versi cepat: teks identik hanya diproses sekali, stem per kata unik. Kembalikan list."""
    slang = dict(SLANG)
    if slang_csv:
        import pandas as pd
        lex = pd.read_csv(slang_csv)
        slang.update(dict(zip(lex["slang"].astype(str), lex["formal"].astype(str))))
    texts = list(texts)
    cache = {t: preprocess(t, preset, slang_dict=slang, **override) for t in {x if x == x else None for x in texts}}
    out = [cache[t if t == t else None] for t in texts]
    if stem:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        st = StemmerFactory().create_stemmer()
        m = {w: st.stem(w) for w in {w for d in out for w in d.split()}}
        out = [" ".join(m[w] for w in d.split()) for d in out]
    return out

def text_versions(df, text="text", slang_csv=None):
    """Tiga versi agar efek normalisasi bisa diuji: text_raw, text_clean (makna utuh), text_norm (slang dinormalkan)."""
    d = df.copy()
    d["text_raw"] = d[text]
    d["text_clean"] = preprocess_corpus(d[text], "tfidf")
    d["text_norm"] = preprocess_corpus(d[text], "social", slang_csv=slang_csv)
    return d

# ===== jalankan =====
docs = preprocess_corpus(df[TEXT], "topic")


## Ekstraksi

TF-IDF per dokumen dan isu korpus.

In [ ]:
import re
from collections import Counter
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer

def tfidf_keywords(docs, k=5, ngram=(1, 3), stop=None, min_df=2):
    v = TfidfVectorizer(ngram_range=ngram, min_df=min_df, max_df=.7, stop_words=stop, sublinear_tf=True)
    X = v.fit_transform(docs).tocsr()
    n = v.get_feature_names_out()
    return [[n[j] for j in X[i].indices[X[i].data.argsort()[::-1][:k]]] for i in range(X.shape[0])]

def ctfidf_terms(docs, labels, top=10, stop=None, min_df=2):
    """Kata khas per kelompok: {label: [kata]}. Abaikan label -1."""
    labels = np.asarray(labels)
    ids = [l for l in np.unique(labels) if l != -1]
    joined = [" ".join(str(d) for d, a in zip(docs, labels) if a == l) for l in ids]
    v = CountVectorizer(stop_words=stop, min_df=1, token_pattern=r"(?u)\b[^\d\W]{2,}\b")
    X = TfidfTransformer(sublinear_tf=True).fit_transform(v.fit_transform(joined)).toarray()
    n = v.get_feature_names_out()
    return {l: [n[j] for j in X[i].argsort()[::-1][:top]] for i, l in enumerate(ids)}

def rake_keywords(doc, stop, k=5, max_len=3):
    w = re.findall(r"\w+", str(doc).lower())
    phrases, cur = [], []
    for x in w:
        if x in stop:
            phrases.append(cur); cur = []
        else:
            cur.append(x)
    phrases = [p for p in phrases + [cur] if 0 < len(p) <= max_len]
    freq, deg = Counter(), Counter()
    for p in phrases:
        for x in p:
            freq[x] += 1; deg[x] += len(p)
    sc = {" ".join(p): sum(deg[x] / freq[x] for x in p) for p in phrases}
    return sorted(sc, key=sc.get, reverse=True)[:k]

def mmr_select(doc_vec, cand_vecs, cands, k=5, diversity=.5):
    """KeyBERT-style. Semua vektor L2-normalized."""
    sim = cand_vecs @ doc_vec
    picked = [int(sim.argmax())]
    while len(picked) < min(k, len(cands)):
        red = (cand_vecs @ cand_vecs[picked].T).max(1)
        sc = (1 - diversity) * sim - diversity * red
        sc[picked] = -9
        picked.append(int(sc.argmax()))
    return [cands[i] for i in picked]

# ===== jalankan =====
kw = tfidf_keywords(docs, K, ngram=(1, 2))
print("isu korpus:", ctfidf_terms(docs, np.zeros(len(docs), int), 20)[0])
for t, k in list(zip(df[TEXT], kw))[:5]:
    print(k, "|", str(t)[:70])


## Output

Satu baris per dokumen.

In [ ]:
df["keywords"] = ["; ".join(x) for x in kw]
df.to_csv("submission.csv", index=False)
